# PHẦN 1: PHÉP KHỬ GAUSS VÀ CÁC ỨNG DỤNG

**Môn học:** Toán Ứng Dụng và Thống Kê (MTH00051)  
**Đồ án 1:** Ma Trận và Cơ Sở của Tính Toán Khoa Học  

<div style="border-left: 5px solid #2196F3; background-color: #E3F2FD; padding: 15px; border-radius: 5px; color: #0D47A1; margin-bottom: 15px;">
<b>📌 TÓM TẮT MỤC TIÊU VÀ PHƯƠNG PHÁP</b><br>
Phần báo cáo này trình bày kết quả cài đặt và thực thi bộ thư viện tính toán ma trận "từ con số 0" (from scratch) bằng Python. Trọng tâm là Phép khử Gauss với kỹ thuật <b>Partial Pivoting</b> nhằm đảm bảo tính ổn định số học.
<br><br>
<b>Các bài toán được giải quyết:</b>
<ol>
    <li><b>Giải hệ phương trình <i>Ax = b</i>:</b> Sử dụng Khử Gauss và Thế ngược.</li>
    <li><b>Tính định thức <i>det(A)</i>:</b> Dựa trên tích đường chéo ma trận tam giác.</li>
    <li><b>Ma trận nghịch đảo <i>A⁻¹</i>:</b> Áp dụng phương pháp Gauss-Jordan.</li>
    <li><b>Hạng và Cơ sở:</b> Xác định hạng, cơ sở không gian dòng, cột và null space.</li>
</ol>
<i>* Tất cả kết quả đều được đối chiếu chéo (validation) với thư viện chuẩn <b>NumPy</b> và <b>SciPy</b>.</i>
</div>



Báo cáo thực thi (Demo) này trình bày kết quả chạy các thuật toán đại số tuyến tính được cài đặt từ đầu (from scratch) bằng Python. Kết quả của thuật toán sẽ được đối chiếu và kiểm chứng trực tiếp với các hàm chuẩn từ thư viện `numpy` và `scipy` để đảm bảo tính chính xác.

### 1.1. Khởi tạo môi trường và Thư viện
Trong phần này, chúng ta nạp các công cụ cần thiết cho quá trình thực nghiệm:
* `numpy` & `scipy`: Hỗ trợ kiểm chứng (validation) kết quả tính toán.
* `fractions`: Hỗ trợ tính toán và hiển thị dưới dạng phân số để triệt tiêu sai số dấu phẩy động.
* Các module `gaussian`, `determinant`, `inverse`, `rank_basis`: Chứa thuật toán do nhóm tự cài đặt.

In [2]:
import numpy as np
from scipy.linalg import null_space
from fractions import Fraction

# Import các hàm từ source code đã viết
from gaussian import gaussian_eliminate, verify_solution
from determinant import determinant
from inverse import inverse
from rank_basis import rank_and_basis

# Thiết lập in số thập phân của numpy cho dễ nhìn
np.set_printoptions(suppress=True, precision=5)

### 1.2. Giải Hệ Phương Trình Tuyến Tính (Khử Gauss với Partial Pivoting)

**Mục tiêu:** Kiểm tra khả năng giải hệ phương trình và tự động hoán đổi dòng (Partial Pivoting) khi phần tử chỉ huy (pivot) bị triệt tiêu (bằng 0).

**Thiết lập thực nghiệm:**
Chạy qua 5 bài test bao phủ các trường hợp cốt lõi: nghiệm duy nhất, vô nghiệm, vô số nghiệm, nghiệm phân số và đặc biệt là hệ cần hoán vị hàng (cố tình đặt $A_{0,0} = 0$).

In [4]:
test_cases_gauss = [
    {"name": "Hệ có nghiệm duy nhất", "A": [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]], "b": [8, -11, -3]},
    {"name": "Hệ vô nghiệm", "A": [[1, 1], [1, 1]], "b": [2, 3]},
    {"name": "Hệ có vô số nghiệm", "A": [[1, 2], [2, 4]], "b": [3, 6]},
    {"name": "Hệ có nghiệm phân số", "A": [[2, 4, 2], [4, 2, 2], [2, 2, 4]], "b": [2, 4, 6]},
    {"name": "Hệ cần hoán vị hàng", "A": [[0, 2, 3], [4, 5, 6], [7, 8, 0]], "b": [13, 32, 23]}
]

for idx, test in enumerate(test_cases_gauss, 1):
    print(f"-------------- Test case {idx}: {test['name']} --------------")
    A = test["A"]
    b = test["b"]
    print("Ma trận hệ số (A):")
    for row in A:
        print(f"  {row}")
    print("Vector hằng số (b):")
    for val in b:
        print(f"  [{val}]")
    # Chạy thuật toán
    print("-> Chạy thuật toán Khử Gauss:")
    result = gaussian_eliminate([row[:] for row in A], b[:])

    if result is not None:
        M, x, s = result
        print(f"Số lần hoán vị hàng (s): {s}")
        print(f"Nghiệm x (Fraction): {x}")
        verify_solution(A, x, b) # Gọi hàm verify

        # So sánh chéo với Numpy (chỉ dành cho hệ có nghiệm duy nhất)
        if x is not None and len(x) > 0 and not isinstance(x[0], dict):
            try:
                x_np = np.linalg.solve(A, b)
                x_float = [float(v) for v in x]
                print(f"Khớp với NumPy? {np.allclose(x_float, x_np)}")
            except np.linalg.LinAlgError:
                pass
        print("\n")

-------------- Test case 1: Hệ có nghiệm duy nhất --------------
Ma trận hệ số (A):
  [2, 1, -1]
  [-3, -1, 2]
  [-2, 1, 2]
Vector hằng số (b):
  [8]
  [-11]
  [-3]
-> Chạy thuật toán Khử Gauss:
  x1 = 2
  x2 = 3
  x3 = -1
Số lần hoán vị hàng (s): 2
Nghiệm x (Fraction): [Fraction(2, 1), Fraction(3, 1), Fraction(-1, 1)]
-> Verify OK: Nghiem duy nhat CHINH XAC.
Khớp với NumPy? True


-------------- Test case 2: Hệ vô nghiệm --------------
Ma trận hệ số (A):
  [1, 1]
  [1, 1]
Vector hằng số (b):
  [2]
  [3]
-> Chạy thuật toán Khử Gauss:
He phuong trinh Vo nghiem.
Số lần hoán vị hàng (s): 0
Nghiệm x (Fraction): None
Khong co nghiem de kiem chung.


-------------- Test case 3: Hệ có vô số nghiệm --------------
Ma trận hệ số (A):
  [1, 2]
  [2, 4]
Vector hằng số (b):
  [3]
  [6]
-> Chạy thuật toán Khử Gauss:
He phuong trinh co Vo so nghiem. Nghiem tong quat:
  x1 = 3 - 2*t2
  x2 = t2
Số lần hoán vị hàng (s): 1
Nghiệm x (Fraction): [{'const': Fraction(3, 1), 1: Fraction(-2, 1)}, {'const': Fra

### 1.3. Tính Định Thức Qua Khử Gauss

**Cơ sở toán học:**
Thay vì dùng công thức khai triển Laplace có độ phức tạp $\mathcal{O}(n!)$ rất chậm, ta áp dụng tính chất của phép biến đổi sơ cấp trên hàng. Quá trình khử Gauss biến ma trận vuông $A$ thành ma trận tam giác trên $U$. Định thức của $A$ được tính bằng tích các phần tử trên đường chéo chính của $U$, nhân với hệ số điều chỉnh dấu nếu có hoán vị hàng:
$$\det(A) = (-1)^s \prod_{i=1}^{n} u_{ii}$$
*(Trong đó: $s$ là số lần hoán đổi hàng trong quá trình Partial Pivoting).*

**Thiết lập thực nghiệm:**
Đánh giá độ chính xác qua 5 bài test: Ma trận vuông thường, ma trận suy biến (phải ra $\det = 0$), và ma trận đơn vị.

In [5]:
test_cases_det = [
    ("CASE 1: Ma trận 2x2", [[1, 2], [3, 4]]),
    ("CASE 2: Ma trận 3x3", [[6, 1, 1], [4, -2, 5], [2, 8, 7]]),
    ("CASE 3: Ma trận suy biến (DET=0)", [[1, 2, 3], [4, 5, 6], [5, 7, 9]]),
    ("CASE 4: Ma trận đơn vị", [[1, 0, 0], [0, 1, 0], [0, 0, 1]]),
    ("CASE 5: Ma trận 4x4", [[1, 0, 2, -1], [3, 0, 0, 5], [2, 1, 4, -3], [1, 0, 5, 0]])
]

for name, A in test_cases_det:
    print(f"-------------- {name} --------------")
    print("Ma trận A:")
    for row in A:
      print(f"  {row}")
    det_frac = determinant(A)
    det_np = np.linalg.det(A)

    print(f"Định thức (của hàm tự định nghĩa): {det_frac} ~ {float(det_frac)}")
    print(f"Định thức (NumPy)  : {det_np:.5f}")
    print(f"-> Khớp kết quả? {np.isclose(float(det_frac), det_np)}")
    print("\n")

-------------- CASE 1: Ma trận 2x2 --------------
Ma trận A:
  [1, 2]
  [3, 4]
Định thức (của hàm tự định nghĩa): -2 ~ -2.0
Định thức (NumPy)  : -2.00000
-> Khớp kết quả? True


-------------- CASE 2: Ma trận 3x3 --------------
Ma trận A:
  [6, 1, 1]
  [4, -2, 5]
  [2, 8, 7]
Định thức (của hàm tự định nghĩa): -306 ~ -306.0
Định thức (NumPy)  : -306.00000
-> Khớp kết quả? True


-------------- CASE 3: Ma trận suy biến (DET=0) --------------
Ma trận A:
  [1, 2, 3]
  [4, 5, 6]
  [5, 7, 9]
Định thức (của hàm tự định nghĩa): 0 ~ 0.0
Định thức (NumPy)  : 0.00000
-> Khớp kết quả? True


-------------- CASE 4: Ma trận đơn vị --------------
Ma trận A:
  [1, 0, 0]
  [0, 1, 0]
  [0, 0, 1]
Định thức (của hàm tự định nghĩa): 1 ~ 1.0
Định thức (NumPy)  : 1.00000
-> Khớp kết quả? True


-------------- CASE 5: Ma trận 4x4 --------------
Ma trận A:
  [1, 0, 2, -1]
  [3, 0, 0, 5]
  [2, 1, 4, -3]
  [1, 0, 5, 0]
Định thức (của hàm tự định nghĩa): 30 ~ 30.0
Định thức (NumPy)  : 30.00000
-> Khớp kết quả? Tr

### 1.4. Tìm Ma Trận Nghịch Đảo (Phương pháp Gauss-Jordan)

**Cơ sở toán học:**
Để tìm ma trận nghịch đảo $A^{-1}$, ta thiết lập một ma trận bổ sung kích thước $n \times 2n$ bằng cách ghép ma trận $A$ với ma trận đơn vị $I$. Bằng cách áp dụng phép khử Gauss-Jordan (khử cả trên và dưới đường chéo chính), ta biến đổi phần bên trái thành ma trận đơn vị. Khi đó, phần bên phải sẽ tự động trở thành ma trận nghịch đảo:
$$[A | I] \xrightarrow{\text{Gauss-Jordan}} [I | A^{-1}]$$

**Thiết lập thực nghiệm:**
Thuật toán sẽ tự động kiểm tra tính suy biến ($\det = 0$) để dừng sớm. Sau khi tìm được $A^{-1}$, ta sẽ thực hiện phép nhân $A \times A^{-1}$ để kiểm chứng xem kết quả có thực sự trả về ma trận đơn vị $I$ hay không.

In [6]:
test_cases_inv = [
    ("CASE 1: Ma trận 2x2", [[4, 7], [2, 6]]),
    ("CASE 2: Ma trận 3x3", [[1, 2, 3], [0, 1, 4], [5, 6, 0]]),
    ("CASE 3: Ma trận đơn vị", [[1, 0], [0, 1]]),
    ("CASE 4: Ma trận suy biến", [[1, 2], [2, 4]]),
    ("CASE 5: Ma trận 1x1", [[5]])
]

for name, A in test_cases_inv:
    print(f"-------------- {name} --------------")

    print("Ma trận A:")
    for row in A:
      print(f"  {row}")

    inv_res = inverse(A)

    if inv_res:
        # Chuyển kết quả (mảng chuỗi phân số) thành mảng số thực để test với numpy
        inv_float = np.array([[float(Fraction(val)) for val in row] for row in inv_res])
        print("Ma trận nghịch đảo thu được:")
        print(inv_float)

        # Kiểm chứng
        inv_np = np.linalg.inv(A)
        identity_check = np.dot(A, inv_float)
        print(f"-> Khớp với NumPy? {np.allclose(inv_float, inv_np)}")
        print(f"-> A * A^-1 có ra ma trận đơn vị I? {np.allclose(identity_check, np.eye(len(A)))}")
    print("\n")

-------------- CASE 1: Ma trận 2x2 --------------
Ma trận A:
  [4, 7]
  [2, 6]
Ma trận nghịch đảo thu được:
[[ 0.6 -0.7]
 [-0.2  0.4]]
-> Khớp với NumPy? True
-> A * A^-1 có ra ma trận đơn vị I? True


-------------- CASE 2: Ma trận 3x3 --------------
Ma trận A:
  [1, 2, 3]
  [0, 1, 4]
  [5, 6, 0]
Ma trận nghịch đảo thu được:
[[-24.  18.   5.]
 [ 20. -15.  -4.]
 [ -5.   4.   1.]]
-> Khớp với NumPy? True
-> A * A^-1 có ra ma trận đơn vị I? True


-------------- CASE 3: Ma trận đơn vị --------------
Ma trận A:
  [1, 0]
  [0, 1]
Ma trận nghịch đảo thu được:
[[1. 0.]
 [0. 1.]]
-> Khớp với NumPy? True
-> A * A^-1 có ra ma trận đơn vị I? True


-------------- CASE 4: Ma trận suy biến --------------
Ma trận A:
  [1, 2]
  [2, 4]
Ma trận suy biến (det = 0) --> không tồn tại ma trận nghịch đảo!


-------------- CASE 5: Ma trận 1x1 --------------
Ma trận A:
  [5]
Ma trận nghịch đảo thu được:
[[0.2]]
-> Khớp với NumPy? True
-> A * A^-1 có ra ma trận đơn vị I? True




### 1.5. Phân Tích Cấu Trúc Ma Trận (Hạng và Cơ Sở)

**Cơ sở toán học:**
* **Hạng (Rank):** Là số lượng dòng khác không (chứa phần tử pivot) của ma trận bậc thang sau khi khử Gauss.
* **Cơ sở không gian dòng (Row Basis):** Trích xuất trực tiếp các dòng chứa pivot từ ma trận bậc thang $U$.
* **Cơ sở không gian cột (Column Basis):** Tìm vị trí các cột chứa pivot trong $U$, sau đó trích xuất các cột ở vị trí tương ứng từ ma trận gốc $A$.
* **Cơ sở không gian nghiệm (Null Space Basis):** Giải hệ thuần nhất $Ax = 0$ bằng thế ngược. Không gian nghiệm được biểu diễn qua các tổ hợp tuyến tính của các biến tự do (free variables).

In [7]:
test_cases_rank = [
    ("CASE 1: Ma trận vuông đủ hạng", [[1, 2, 3], [4, 5, 6], [7, 8, 10]]),
    ("CASE 2: Ma trận phụ thuộc tuyến tính", [[1, 2, 3], [4, 5, 6], [7, 8, 9]]),
    ("CASE 3: Ma trận chữ nhật (2x4)", [[1, 2, 0, 3], [0, 0, 1, 4]]),
    ("CASE 4: Ma trận Zero (rank = 0)", [[0, 0], [0, 0]]),
    ("CASE 5: Ma trận đơn vị", [[1, 0, 0], [0, 1, 0], [0, 0, 1]]),
    ("CASE 6: Ma trận một dòng", [[1, 2, 3, 4]]),
    ("CASE 7: Ma trận một cột", [[1], [2], [3]])
]

for name, A in test_cases_rank:
    print(f"-------------- {name} --------------")
    print("Ma trận A:")
    for row in A:
      print(f"  {row}")

    res = rank_and_basis(A)
    rank_np = np.linalg.matrix_rank(A)

    print(f"Hạng (của hàm tự định nghĩa): {res['rank']} | Hạng (NumPy): {rank_np}")
    print(f"-> Hạng tính đúng? {res['rank'] == rank_np}")

    print(f"Cơ sở dòng: {res['row_basis']}")
    print(f"Cơ sở cột : {res['col_basis']}")
    print(f"Cơ sở Null: {res['null_basis']}")
    print("\n")

-------------- CASE 1: Ma trận vuông đủ hạng --------------
Ma trận A:
  [1, 2, 3]
  [4, 5, 6]
  [7, 8, 10]
Hạng (của hàm tự định nghĩa): 3 | Hạng (NumPy): 3
-> Hạng tính đúng? True
Cơ sở dòng: [['7', '8', '10'], ['0', '6/7', '11/7'], ['0', '0', '-1/2']]
Cơ sở cột : [['1', '4', '7'], ['2', '5', '8'], ['3', '6', '10']]
Cơ sở Null: []


-------------- CASE 2: Ma trận phụ thuộc tuyến tính --------------
Ma trận A:
  [1, 2, 3]
  [4, 5, 6]
  [7, 8, 9]
Hạng (của hàm tự định nghĩa): 2 | Hạng (NumPy): 2
-> Hạng tính đúng? True
Cơ sở dòng: [['7', '8', '9'], ['0', '6/7', '12/7']]
Cơ sở cột : [['1', '4', '7'], ['2', '5', '8']]
Cơ sở Null: [['1', '-2', '1']]


-------------- CASE 3: Ma trận chữ nhật (2x4) --------------
Ma trận A:
  [1, 2, 0, 3]
  [0, 0, 1, 4]
Hạng (của hàm tự định nghĩa): 2 | Hạng (NumPy): 2
-> Hạng tính đúng? True
Cơ sở dòng: [['1', '2', '0', '3'], ['0', '0', '1', '4']]
Cơ sở cột : [['1', '0'], ['0', '1']]
Cơ sở Null: [['-2', '1', '0', '0'], ['-3', '0', '-4', '1']]


-----------

<div style="border-left: 5px solid #4CAF50; background-color: #E8F5E9; padding: 15px; border-radius: 5px; color: #1B5E20; margin-top: 20px;">
<h3 style="margin-top: 0; color: #2E7D32;">💡 KẾT LUẬN THỰC NGHIỆM PHẦN 1</h3>
<ul>
    <li>Thuật toán khử Gauss tự cài đặt xử lý rất mượt các hình thái hệ phương trình khác nhau. Thuật toán thế ngược đã giải quyết triệt để bài toán in ra nghiệm tổng quát có chứa biến tự do.</li>
    <li>Kỹ thuật <b>Partial Pivoting</b> chứng minh tính ổn định xuất sắc khi chủ động đẩy các hàng có giá trị chốt (pivot) lớn nhất lên trên, vượt qua được bài test phần tử chéo bằng 0.</li>
    <li>Việc áp dụng kiểu dữ liệu <code>Fraction</code> giúp hiển thị đáp án trực quan và loại bỏ hoàn toàn sai số làm tròn. Các test case đều đạt kết quả khớp hoàn toàn khi verify chéo với <code>NumPy</code>.</li>
</ul>
</div>